# 4 · Preparación del dataset para el modelado

**Proyecto:** Predicción de readmisión hospitalaria en pacientes diabéticos
**Dataset:** *Diabetes 130-US hospitals for years 1999-2008* (UCI Machine Learning Repository)
**Autor:** Diego Rodríguez Díaz del Campo

---

### Objetivo de este notebook

Convertir el dataset limpio en un conjunto de datos **listo para entrenar un modelo**,
aplicando las decisiones que el EDA dejó preparadas (apartados 5.6, 6.3 y 7 del
notebook 03). Cada transformación se justifica con la evidencia del EDA; nada se hace
por rutina.

El modelado **no** forma parte de este proyecto: este notebook termina con los
conjuntos `train` y `test` guardados en disco, que serán la entrada de un proyecto
futuro.

| | |
|---|---|
| **Entrada** | `data/processed/diabetic_data_clean.csv` (101.766 × 35) y `data/raw/diabetic_data.csv` (solo para recuperar `patient_nbr`) |
| **Variable objetivo** | `readmitted` |
| **Salida** | `data/processed/train.csv` y `data/processed/test.csv` |

### Contenido

1. Configuración, carga y recuperación de `patient_nbr`
2. Exclusión de pacientes fallecidos y en cuidados paliativos
3. Agrupación de categorías poco frecuentes y de alta cardinalidad
4. Eliminación de variables sin señal
5. Formulación de la variable objetivo
6. Codificación de las variables categóricas
7. División en `train` y `test` sin fuga de datos entre pacientes
8. Guardado y conclusiones

## 1 · Configuración, carga y recuperación de `patient_nbr`

### 1.1 Carga y recuperación de tipos

Se parte del dataset limpio (notebook 02). Como se explicó en el notebook 03, el CSV
**no conserva los `dtype`**: los tres códigos administrativos (`admission_type_id`,
`discharge_disposition_id`, `admission_source_id`) y los códigos de diagnóstico vuelven
a leerse como números y hay que declararlos de nuevo como categorías. Se repite aquí la
misma corrección que en el notebook 03, sin volver a discutirla.

Solo se importa lo que este apartado necesita (`pandas`, `numpy`, `pathlib`). Las
librerías gráficas y `scikit-learn` se importarán en el apartado en que hagan falta,
para que quede claro *para qué* se usa cada una.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path(r'D:\Cosas Diego\Universidad\Proyectos personales\proyecto 1')
RAW = BASE / 'data' / 'raw'
PROC = BASE / 'data' / 'processed'

df = pd.read_csv(PROC / 'diabetic_data_clean.csv', low_memory=False)

print(f'Dimensiones: {df.shape}')
df.head()

Dimensiones: (101766, 35)


,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,...,glipizide,glyburide,pioglitazone,rosiglitazone,acarbose,insulin,glyburide-metformin,change,diabetesMed,readmitted
0,Caucasian,Female,[0-10),6,25,1,1,Unknown,Pediatrics-Endocrinology,41,...,No,No,No,No,No,No,No,No,No,NO
1,Caucasian,Female,[10-20),1,1,7,3,Unknown,Unknown,59,...,No,No,No,No,No,Up,No,Ch,Yes,>30
2,AfricanAmerican,Female,[20-30),1,1,7,2,Unknown,Unknown,11,...,Steady,No,No,No,No,No,No,No,Yes,NO
3,Caucasian,Male,[30-40),1,1,7,2,Unknown,Unknown,44,...,No,No,No,No,No,Up,No,Ch,Yes,NO
4,Caucasian,Male,[40-50),1,1,7,1,Unknown,Unknown,51,...,Steady,No,No,No,No,Steady,No,Ch,Yes,NO


In [2]:
# Códigos administrativos: son etiquetas, no cantidades
columnas_categoricas_numericas = [
    'admission_type_id',
    'discharge_disposition_id',
    'admission_source_id'
]

df[columnas_categoricas_numericas] = df[columnas_categoricas_numericas].astype('category')

# Códigos de diagnóstico: el código 700 no es "mayor" que el 200
df[['diag_1', 'diag_2', 'diag_3']] = df[['diag_1', 'diag_2', 'diag_3']].astype('category')

df.dtypes

race                          object
gender                        object
age                           object
admission_type_id           category
discharge_disposition_id    category
admission_source_id         category
time_in_hospital               int64
payer_code                    object
medical_specialty             object
num_lab_procedures             int64
num_procedures                 int64
num_medications                int64
number_outpatient              int64
number_emergency               int64
number_inpatient               int64
diag_1                      category
diag_2                      category
diag_3                      category
number_diagnoses               int64
max_glu_serum                 object
A1Cresult                     object
metformin                     object
repaglinide                   object
nateglinide                   object
glimepiride                   object
glipizide                     object
glyburide                     object
p

### 1.2 Por qué hay que recuperar `patient_nbr`

En la limpieza se eliminó `patient_nbr` porque un identificador de paciente no es una
variable predictora. Sin embargo, el notebook 01 mostró que los 101.766 ingresos
corresponden a **71.518 pacientes distintos**: hay pacientes con 2, 3 y hasta 40
ingresos en el dataset.

Esto tiene una consecuencia directa sobre el apartado 7. Si se dividen las *filas* al
azar entre `train` y `test`, los ingresos de un mismo paciente quedarán repartidos en
ambos conjuntos. El modelo vería en `test` a pacientes que ya conoce de `train`, con sus
mismas características demográficas y clínicas, y su rendimiento parecería mejor de lo
que es en realidad. Es una forma de **fuga de datos** (*data leakage*): la división debe
hacerse **por paciente**, y para eso hace falta el identificador.

### Cómo se recupera: por posición, no por clave

El dataset limpio ya no tiene ninguna columna que sirva de clave para cruzar con el
original (`encounter_id` también se eliminó). Pero la limpieza **eliminó 15 columnas y
0 filas, y nunca reordenó**: la fila *i* del dataset limpio es la fila *i* del original.
Por tanto, `patient_nbr` se puede copiar del original **por posición**.

Es una hipótesis razonable, pero es una hipótesis. Antes de pegar la columna se comprueba
con dos pruebas:

1. Que ambos ficheros tienen exactamente **101.766 filas**.
2. Que una columna que sobrevivió intacta a la limpieza, `time_in_hospital`, **coincide
   fila a fila** en los dos ficheros.

Del original solo se leen las dos columnas necesarias (`usecols`): `patient_nbr`, que es
lo que se quiere recuperar, y `time_in_hospital`, que sirve de columna de control.

In [3]:
original = pd.read_csv(
    RAW / 'diabetic_data.csv',
    usecols=['patient_nbr', 'time_in_hospital']
)

print(f'Filas en el limpio:   {len(df)}')
print(f'Filas en el original: {len(original)}')

Filas en el limpio:   101766
Filas en el original: 101766


Las dos tablas tienen el mismo número de filas. Ahora la comprobación decisiva: que
`time_in_hospital` coincide **posición a posición**.

Se compara con `.to_numpy()` en vez de comparar las dos Series directamente. La razón es
que pandas, al operar entre dos Series, las **alinea por índice**; aquí interesa
justamente lo contrario, comparar por posición pura, porque es lo que se va a asumir al
pegar la columna. Con los arrays de NumPy la comparación es estrictamente posicional.

In [4]:
coinciden = (df['time_in_hospital'].to_numpy() == original['time_in_hospital'].to_numpy())

print(f'Filas que coinciden: {coinciden.sum()} de {len(coinciden)}')
print(f'¿Coinciden todas?    {coinciden.all()}')

Filas que coinciden: 101766 de 101766
¿Coinciden todas?    True


Las dos comprobaciones son favorables, así que se pega `patient_nbr` por posición. Se
inserta como **primera columna** con `insert(0, ...)`, porque es un identificador y no una
variable más: así queda claro a simple vista que no forma parte de las predictoras.

Como verificación final, el número de pacientes distintos debe ser **71.518**, la cifra
obtenida en el notebook 01 sobre el dataset original. Si diera otra cosa, la columna se
habría pegado desalineada.

In [5]:
df.insert(0, 'patient_nbr', original['patient_nbr'].to_numpy())

print(f'Dimensiones:        {df.shape}')
print(f'Pacientes distintos: {df["patient_nbr"].nunique()}')
df.head()

Dimensiones:        (101766, 36)
Pacientes distintos: 71518


,patient_nbr,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,...,glipizide,glyburide,pioglitazone,rosiglitazone,acarbose,insulin,glyburide-metformin,change,diabetesMed,readmitted
0,8222157,Caucasian,Female,[0-10),6,25,1,1,Unknown,Pediatrics-Endocrinology,...,No,No,No,No,No,No,No,No,No,NO
1,55629189,Caucasian,Female,[10-20),1,1,7,3,Unknown,Unknown,...,No,No,No,No,No,Up,No,Ch,Yes,>30
2,86047875,AfricanAmerican,Female,[20-30),1,1,7,2,Unknown,Unknown,...,Steady,No,No,No,No,No,No,No,Yes,NO
3,82442376,Caucasian,Male,[30-40),1,1,7,2,Unknown,Unknown,...,No,No,No,No,No,Up,No,Ch,Yes,NO
4,42519267,Caucasian,Male,[40-50),1,1,7,1,Unknown,Unknown,...,Steady,No,No,No,No,Steady,No,Ch,Yes,NO


### Interpretación del apartado 1

Las tres verificaciones han sido favorables:

| Comprobación | Resultado esperado | Obtenido |
|---|---|---|
| Filas en ambos ficheros | 101.766 | 101.766 y 101.766 |
| `time_in_hospital` coincide fila a fila | todas | 101.766 de 101.766 (`True`) |
| Pacientes distintos tras pegar `patient_nbr` | 71.518 (notebook 01) | 71.518 |

La hipótesis de que la limpieza no alteró el orden de las filas queda **confirmada con
evidencia**, no asumida. El dataset de trabajo tiene ahora **101.766 × 36**: las 35
columnas del limpio más `patient_nbr` en primera posición.

Dos advertencias para el resto del notebook:

- `patient_nbr` es un **identificador**, no una variable predictora. Se conserva
  únicamente para hacer la división por paciente del apartado 7 y **no** debe entrar
  en el modelo: un número de historia clínica no dice nada del riesgo de readmisión.
- La cifra de 71.518 pacientes es válida **ahora**; cuando en el apartado 2 se excluyan
  los fallecidos y los pacientes en cuidados paliativos, cambiará y habrá que
  recalcularla.